<a href="https://colab.research.google.com/github/ZaychenkoNA/-Algorithms-and-data-structures-VK-/blob/main/HW_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right
        self.balance_factor = 0

**1. Полное бинарное дерево**

Реализуйте функцию, которая проверяет, является ли заданное бинарное дерево
полным.

Необходимо корректно определить условие заполненности уровней и объяснить
выбранный способ обхода (например, BFS).


In [ ]:
from collections import deque

def isCompleteTree(root):
    if root is None:
        return True

    queue = deque([root])
    found_non_full = False
    while queue:
        node = queue.popleft()
        if node.left:
            if found_non_full:
                return False
            queue.append(node.left)
        else:
            found_non_full = True
        if node.right:
            if found_non_full:
                return False
            queue.append(node.right)
        else:
            found_non_full = True

    return True

BFS гарантирует порядок обхода: сначала все узлы уровня k, затем уровня
k+1, причём в пределах уровня — строго слева направо.

Это позволяет однозначно обнаружить первый "пропуск" — момент, когда у узла отсутствует потомок, но при этом в очереди ещё есть узлы с потомками.

In [3]:
# Реализация основных функций мин-кучи для следующих задач
def sift_down(heap, i, n):
    while True:
        smallest = i
        left = 2 * i + 1
        right = 2 * i + 2

        if left < n and heap[left] < heap[smallest]:
            smallest = left
        if right < n and heap[right] < heap[smallest]:
            smallest = right

        if smallest == i:
            break

        heap[i], heap[smallest] = heap[smallest], heap[i]
        i = smallest

def sift_up(heap, i):
    while i > 0:
        parent = (i - 1) // 2
        if heap[i] < heap[parent]:
            heap[i], heap[parent] = heap[parent], heap[i]
            i = parent
        else:
            break

def build_heap(arr):
    n = len(arr)
    for i in range(n // 2 - 1, -1, -1):
        sift_down(arr, i, n)
    return arr


def pop(heap):
    if not heap:
        raise IndexError
    n = len(heap)
    if n == 1:
        return heap.pop()


    root = heap[0]
    heap[0] = heap[-1]
    heap.pop()
    sift_down(heap, 0, len(heap))
    return root

def push(heap, item):
    heap.append(item)
    sift_up(heap, len(heap) - 1)

**2. Объединение K отсортированных массивов**

Напишите функцию, объединяющую K отсортированных массивов в один
отсортированный массив.

Для эффективного решения используйте мин-кучу и проанализируйте сложность
алгоритма.


In [4]:
def merge_k_sorted(arrays):
    if not arrays:
        return []
    heap = []

    for i, arr in enumerate(arrays):
        if arr:
            push(heap, (arr[0], i, 0))

    result = []
    while heap:
        val, arr_idx, elem_idx = pop(heap)
        result.append(val)
        next_elem_idx = elem_idx + 1
        if next_elem_idx < len(arrays[arr_idx]):
            push(heap, (arrays[arr_idx][next_elem_idx], arr_idx, next_elem_idx))

    return result

**Сложность по времени:**

Инициализация кучи:

K операций push, каждая O(logK). Итого O(KlogK).

Основной цикл: выполняется ровно N раз. На каждой итерации:
pop — O(logK),
push — O(logK).

Итого: O(NlogK).


**3. K-ый наименьший / наибольший элемент**

Дан массив чисел. Необходимо найти K-ый наименьший или K-ый наибольший
элемент.

Рассмотрите эффективные подходы (например, с использованием кучи) и обоснуйте
сложность решения.

In [1]:
def kthLargestArray(arr, k):
    if not arr:
        raise ValueError("Массив не может быть пустым")
    if k < 1 or k > len(arr):
        raise ValueError(f"K должно быть в диапазоне от 1 до {len(arr)}")
    heap = build_heap([-x for x in arr]) # Нужна макс-куча. Сделаем мин-кучу из противоположных по знаку элементов.
    for i in range(k):
        root = pop(heap)
    return -root



Создаем макс-кучу из массива за О(N). Вытаскиваем вершину до тех пор, пока не дойдем до к-ого элемента: К операций по О(logN) каждая. Итого О(N + KlogN) по времени и О(1) по памяти.

In [ ]:
def heapreplace(heap, item):
    returnitem = heap[0]
    heap[0] = item
    sift_down(heap, 0, len(heap))
    return returnitem

def kthSmallest(arr, k):
    if not arr or k < 1 or k > len(arr):
        raise ValueError("Некорректные входные данные: массив пуст или K вне диапазона")

    heap = [-x for x in arr[:k]]
    build_heap(heap)
    for num in arr[k:]:
        if num < -heap[0]:
            heapreplace(heap, num)

    return -heap[0]

Создаем макс-кучу из первых К элементов массива. Далее проходим по остальным элементам массива: если он меньше корня - меняем корень на этот элемент и восстанавливаем кучу. В итоге получаем макс-кучу из К наименьших элементов. Значит нужный нам находится в корне.

heapreplace быстрее комбинации push и pop т.к. проходим всего один раз по куче, вместо двух.

О(К) - создание кучи

(N - K)logK - проход по остальному массиву

Итого: O(NlogK) по времени (N >= K), О(K) по памяти. Лучше предыдущего алгоритма, когда N >> K.

**4. K-ый наименьший / наибольший элемент в BST**

Дано бинарное дерево поиска.

Реализуйте алгоритм нахождения K-ого наименьшего или наибольшего элемента,
используя свойства BST и обход дерева.


In [ ]:
def kthSmallestBST(root, k):
    stack = []
    current = root
    count = 0
    while stack or current:
        while current:
            stack.append(current)
            current = current.left

        current = stack.pop()
        count += 1
        if count == k:
            return current.val

        current = current.right

    return None

def kthLargestBST(root, k):
    stack = []
    current = root
    count = 0
    while stack or current:
        while current:
            stack.append(current)
            current = current.right

        current = stack.pop()
        count += 1
        if count == k:
            return current.val

        current = current.left

    return None

**5. Balance factor**

На вход функции передаётся указатель на корень бинарного дерева поиска.

Необходимо для каждого узла вычислить и заполнить поле balance_factor (разницу
высот левого и правого поддеревьев), объяснив способ вычисления высоты.


In [ ]:
def compute_balance_factors(root):
    if root is None:
        return 0

    h_left = compute_balance_factors(root.left)
    h_right = compute_balance_factors(root.right)

    root.balance_factor = h_left - h_right
    return 1 + max(h_left, h_right)



Рекурсивно получаем высоту левого и правого поддерева и считаем баланс-фактор для каждого узла.

**6. Преобразование в зеркальное дерево**

Реализуйте алгоритм зеркального отражения бинарного дерева, меняя местами
левые и правые поддеревья каждого узла.

Поясните, почему алгоритм корректен и какова его временная сложность.

In [ ]:
# Через рекурсию
def mirrorTree(root):

    if root is None:
        return None

    root.left, root.right = root.right, root.left

    mirrorTree(root.left)
    mirrorTree(root.right)

    return root

In [ ]:
# Через обход в ширину
from collections import deque

def mirrorTree_bfs(root):
    if root is None:
        return None

    queue = deque([root])

    while queue:
        node = queue.popleft()

        node.left, node.right = node.right, node.left

        if node.left:
            queue.append(node.left)
        if node.right:
            queue.append(node.right)

    return root

На каждом шаге мы меняем левого и правого потомка местами и так повторяем для каждого узла (вне зависимости от подхода).
Сложность для каждого подхода О(N) - проходим по каждому узлу. В каждой итерации константное число операций.